# Пример - Проверка структурных тождеств

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **73ac82d**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/73ac82d5c16afbfd47285436e1b6f37d5a68f8af.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Пять расчётов сопровождают пять заметок серии. Проверим равенство левой и правой частей, найдём независимые отношения и посмотрим, как замена формы влияет на расчёт сочленения.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 73ac82d](https://github.com/magin67/polyform_algebra/commit/73ac82d5c16afbfd47285436e1b6f37d5a68f8af).


## Тождество тетраэдра

На вершинах $a,b,c,d$ слева сложим формы трёх разбиений на пары, справа - формы четырёх троек. Функция `q` строит квадрат произведения границ перечисленных блоков. Пропущенные вершины считаются одноэлементными блоками. Строка `"ab"` здесь обозначает блок из двух буквенных вершин.

$$
q_{ab|cd}+q_{ac|bd}+q_{ad|bc}
=q_{abc}+q_{abd}+q_{acd}+q_{bcd}
$$

Проверка сравнивает полные формальные коэффициенты в одном контексте. Затем увеличим на единицу коэффициент при $q_{abc}$ справа. Проверка должна обнаружить ошибку, а в разности сторон появится ненулевой коэффициент. Выведенные базисные объекты $a\wedge b$ относятся к внешней степени пространства формальных вершин.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0/%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D0%B9-%D0%B8-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%BE-%D1%82%D0%B5%D1%82%D1%80%D0%B0%D1%8D%D0%B4%D1%80%D0%B0#%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%BE-%D1%82%D0%B5%D1%82%D1%80%D0%B0%D1%8D%D0%B4%D1%80%D0%B0).

In [3]:
from itertools import combinations

from sympy import Matrix, zeros
from polyform_algebra.core import (
    Context, equivalent, form_coefficients, form_exp, form_product, grade_part,
    network, partition_form, spanning_coefficient, spanning_tree_weight,
)

def q(context, *blocks):
    return partition_form(context, tuple(tuple(block) for block in blocks)).value

def tetrahedron(context, vertices):
    a, b, c, d = vertices
    left = q(context, (a, b), (c, d)) + q(context, (a, c), (b, d)) + q(context, (a, d), (b, c))
    right = q(context, (a, b, c)) + q(context, (a, b, d)) + q(context, (a, c, d)) + q(context, (b, c, d))
    return left, right

four = Context(tuple("abcd"))
left, right = tetrahedron(four, four.labels)
assert equivalent(left, right).value
assert form_coefficients(left - right).value == ()

perturbed = right + q(four, "abc")
assert not equivalent(left, perturbed).value
(i, j), coefficient = form_coefficients(left - perturbed).value[0]
blades = tuple(" ∧ ".join(four.labels[position] for position in blade) for blade in (i, j))
print("Tetrahedron identity:", equivalent(left, right).value)
print("First coefficient of left - perturbed right:", blades, "→", coefficient)

Tetrahedron identity: True
First coefficient of left - perturbed right: ('a ∧ b', 'a ∧ b') → -1


## Перенос и умножение

Перенесём тождество в контекст $a,b,c,d,e,f$ и умножим обе стороны на форму отдельной пары $ef$. Второе равенство проверим после увеличения четвёртого блока до $def$, как в формуле (5) заметки.

На пяти вершинах проверим также $E_5[2,1]=3E_5[3,0]$. Слева каждую тройку дополняет одна пара, справа каждая четвёрка задаёт одно разбиение. Каждое разбиение учитывается один раз. Этот расчёт связывает структурные тождества с симметрическими суммами.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0/%D0%BA%D0%B0%D0%BA-%D0%BF%D0%BE%D0%BB%D1%83%D1%87%D0%B0%D1%82%D1%8C-%D0%BD%D0%BE%D0%B2%D1%8B%D0%B5-%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0#%D0%B4%D0%B2%D0%B0-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D0%B0-%D0%BD%D0%B0-%D1%88%D0%B5%D1%81%D1%82%D0%B8-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [4]:
six = Context(tuple("abcdef"))
left, right = tetrahedron(six, "abcd")
extra_pair = q(six, "ef")
lifted_left = form_product(left, extra_pair).value
lifted_right = form_product(right, extra_pair).value
assert equivalent(lifted_left, lifted_right).value

enlarged_left = q(six, "abc", "def")
enlarged_right = (
    q(six, "ab", "cdef") + q(six, "ac", "bdef") + q(six, "bc", "adef")
    - q(six, "bcdef") - q(six, "acdef") - q(six, "abdef")
)
assert equivalent(enlarged_left, enlarged_right).value
print("With the extra pair ef:", equivalent(lifted_left, lifted_right).value)
print("With the larger block def:", equivalent(enlarged_left, enlarged_right).value)

five = Context(tuple("abcde"))
triple_pair = [
    q(five, triple, tuple(v for v in five.labels if v not in triple))
    for triple in combinations(five.labels, 3)
]
quadruple = [q(five, block) for block in combinations(five.labels, 4)]
E_21 = sum(triple_pair[1:], triple_pair[0])
E_30 = sum(quadruple[1:], quadruple[0])
assert equivalent(E_21, 3 * E_30).value
print("E5[2,1] = 3 E5[3,0]:", equivalent(E_21, 3 * E_30).value)

With the extra pair ef: True
With the larger block def: True
E5[2,1] = 3 E5[3,0]: True


## Ранг и ядро на четырёх вершинах

Расположим в первых шести столбцах формы непересекающихся разбиений, а последней поставим $q_{ac|bd}$. Строка соответствует одной паре базисных внешних объектов. Раскрываем каждую форму один раз и используем точное приведение матрицы к ступенчатому виду. Первые шесть столбцов опорные, поэтому ранг равен $6$, а ядро одномерно. Вектор ядра с последней координатой $1$ выражает пересекающееся разбиение через базис.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0/%D0%B1%D0%B0%D0%B7%D0%B8%D1%81-%D0%B8-%D1%80%D0%B0%D0%B7%D0%BC%D0%B5%D1%80%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0-%D1%84%D0%BE%D1%80%D0%BC-%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D0%B9#%D1%87%D0%B5%D1%82%D1%8B%D1%80%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%B1%D0%B0%D0%B7%D0%B8%D1%81-%D0%B8-%D1%8F%D0%B2%D0%BD%D0%B0%D1%8F-%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B0).

In [5]:
def coefficient_matrix(forms):
    columns = [dict(form_coefficients(form).value) for form in forms]
    pairs = sorted(set().union(*columns))
    return Matrix([[column.get(pair, 0) for column in columns] for pair in pairs])

basis = [
    q(four, "bcd"), q(four, "acd"), q(four, "abd"), q(four, "abc"),
    q(four, "ab", "cd"), q(four, "ad", "bc"),
]
crossing = q(four, "ac", "bd")
matrix = coefficient_matrix(basis + [crossing])
reduced, pivots = matrix.rref()
relation = Matrix([-reduced[row, 6] for row in range(6)] + [1])
assert pivots == tuple(range(6))
assert tuple(relation) == (-1, -1, -1, -1, 1, 1, 1)
assert matrix * relation == zeros(matrix.rows, 1)
print("Rank:", len(pivots), "| Nullity:", matrix.cols - len(pivots))
print("Relation coefficients:", tuple(relation))

Rank: 6 | Nullity: 1
Relation coefficients: (-1, -1, -1, -1, 1, 1, 1)


## Смена базиса при сочленении

Сочленим пути $a-c-b-d$ и $b-a-d-c$ с единичными проводимостями. Их объединение даёт $K_4$. В $P_1$ перепишем второй грейд по тождеству тетраэдра, устранив $q_{ac|bd}$. Сначала сравним формы второго грейда, затем оба произведения с $P_2$. Коэффициенты $1,7,7,1$ дают $\tau(K_4)=16$. Для отдельной сверки вычислим остовной коэффициент по лапласиану полного графа.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2#%D1%87%D0%B5%D1%82%D1%8B%D1%80%D0%B5-%D0%BE%D0%B1%D1%89%D0%B8%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%B8-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%BE-%D1%82%D0%B5%D1%82%D1%80%D0%B0%D1%8D%D0%B4%D1%80%D0%B0).

In [6]:
P1 = form_exp(q(four, "ac") + q(four, "bc") + q(four, "bd")).value
P2 = form_exp(q(four, "ab") + q(four, "ad") + q(four, "cd")).value
before = grade_part(P1, 2).value
after = (
    2 * q(four, "bcd") + q(four, "acd") + q(four, "abd") + 2 * q(four, "abc")
    - q(four, "ab", "cd") - q(four, "ad", "bc")
)
assert equivalent(before, after).value
rewritten_P1 = P1 - before + after
original_product = form_product(P1, P2).value
rewritten_product = form_product(rewritten_P1, P2).value
assert equivalent(original_product, rewritten_product).value

contributions = tuple(
    spanning_coefficient(form_product(grade_part(P1, k).value, grade_part(P2, 3 - k).value).value).value
    for k in range(4)
)
complete = network(four.labels, tuple((a, b, 1) for a, b in combinations(four.labels, 2)))
assert contributions == (1, 7, 7, 1)
assert spanning_coefficient(rewritten_product).value == spanning_tree_weight(complete).value == 16
print("Grade contributions:", tuple(map(int, contributions)), "| tau(K4) =", sum(contributions))

Grade contributions: (1, 7, 7, 1) | tau(K4) = 16


## Носители на пяти вершинах

На пяти вершинах есть пять четырёхвершинных наборов. На каждом проверим левую и правую части тождества тетраэдра, оставив пятую вершину одноэлементным блоком. В каждом отношении своё пересекающееся разбиение, которое отсутствует в остальных четырёх. Проверим это по коэффициентам символов разбиений до раскрытия.

Затем сравним с ядром матрицы раскрытых коэффициентов всех $25$ форм второго грейда. Ранг равен $20$, поэтому пять проверенных независимых отношений дают всё ядро в этом примере.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0/%D0%BD%D0%BE%D1%81%D0%B8%D1%82%D0%B5%D0%BB%D0%B8-%D1%81%D1%82%D1%80%D1%83%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D1%85-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2#%D1%87%D0%B5%D1%82%D1%8B%D1%80%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%BE%D0%B4%D0%BD%D0%BE-%D0%BE%D1%82%D0%BD%D0%BE%D1%88%D0%B5%D0%BD%D0%B8%D0%B5).

In [7]:
supports = tuple(combinations(five.labels, 4))
all_forms = [q(five, triple) for triple in combinations(five.labels, 3)]
relations = []
crossing_terms = []
for a, b, c, d in supports:
    left, right = tetrahedron(five, (a, b, c, d))
    assert equivalent(left, right).value
    relations.append(dict((left - right).terms))
    crossing = q(five, (a, c), (b, d))
    crossing_terms.append(crossing.terms[0][0])
    all_forms.extend((q(five, (a, b), (c, d)), crossing, q(five, (a, d), (b, c))))

# These are coefficients of partition symbols before exterior expansion.
# Each relation has its own crossing partition, so the five relations are independent.
for row, relation in enumerate(relations):
    assert [relation.get(term, 0) for term in crossing_terms] == [int(row == i) for i in range(5)]
matrix = coefficient_matrix(all_forms)
rank = matrix.rank()
assert matrix.cols == 25 and rank == 20 and matrix.cols - rank == len(supports)
print("Four-vertex supports:", ", ".join("".join(support) for support in supports))
print("Forms:", matrix.cols, "| Rank:", rank, "| Independent relations:", matrix.cols - rank)

Four-vertex supports: abcd, abce, abde, acde, bcde
Forms: 25 | Rank: 20 | Independent relations: 5
